# Notebook 04a: Static Class-Mean-Features (CMF) Unlearning (`04a_cmf_static.ipynb`)

### Objectives & Workflow:
1. **CMF Representation Unlearning Framework**: Apply the CMF mechanism across all 5 machine unlearning baselines:
   - **Random Label (RL)**
   - **NegGrad+**
   - **SalUn**
   - **SCRUB**
   - **TarUn / UNSIR**
2. **CMF Head Reconstruction & Freezing**: Reconstruct classifier weights directly from centered & normalized clean retain ($D_R$) class-mean features $\mathbf{w}_c = \frac{\bar{\mathbf{z}}_c - \bar{\mathbf{z}}}{\|\bar{\mathbf{z}}_c - \bar{\mathbf{z}}\|_2}$, freeze the linear head, and optimize only the encoder $\theta$ using each unlearning algorithm.
3. **Re-Tying & Multi-Tier Evaluation**: Reconstruct CMF head on clean $D_R$ after unlearning and compute 3-tier metrics (Output Acc, Linear Probe Acc, NCC Acc) on $D_V$, $D_R$, $D_F$, and $D_T$.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    extract_features,
    evaluate_three_tier_metrics,
    visualize_feature_space_and_boundaries,
    print_metrics_summary_table,
    MODEL_CONFIGS
)

from unlearning_methods import (
    unlearn_random_label,
    unlearn_neggrad_plus,
    unlearn_salun,
    unlearn_scrub,
    unlearn_tarun_unsir
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. CMF Head Reconstruction Utility

In [ ]:
@torch.no_grad()
def reconstruct_cmf_head(model, dataloader, num_classes=10, device="cuda"):
    """
    Computes class-mean features, centers them against the global mean, normalizes, and assigns to linear weights.
    Freezes the linear classifier head in-place.
    """
    feats, targets = extract_features(model.encoder, dataloader, device=device)
    class_means = []
    
    for c in range(num_classes):
        mask = (targets == c)
        if mask.sum() > 0:
            mean = feats[mask].mean(axis=0)
        else:
            mean = np.zeros(feats.shape[1])
        class_means.append(mean)
        
    class_means = np.stack(class_means, axis=0) # [C, D]
    global_mean = class_means.mean(axis=0, keepdims=True)
    centered = class_means - global_mean
    norms = np.linalg.norm(centered, axis=1, keepdims=True) + 1e-8
    normalized_weights = centered / norms # [C, D]
    
    # Assign to model linear layer
    weight_tensor = torch.tensor(normalized_weights, dtype=torch.float32, device=device)
    model.head.linear.weight.data.copy_(weight_tensor)
    if model.head.linear.bias is not None:
        model.head.linear.bias.data.zero_()
    
    # Freeze classifier head
    model.head.linear.weight.requires_grad = False
    if model.head.linear.bias is not None:
        model.head.linear.bias.requires_grad = False

## 2. Static CMF Unlearning Execution Loop Across All 5 Methods

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

METHODS = ["random_label", "neggrad_plus", "salun", "scrub", "tarun_unsir"]

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
from data_loader import load_raw_dataset as _load_raw_dataset
raw_train, raw_test = _load_raw_dataset(DATASET_NAME, root="./data")
if raw_test is None:
    raise ValueError(f"Dataset '{DATASET_NAME}' has no separate test split via load_raw_dataset; handle UTKFace via build_unlearning_splits instead.")

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/plots", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)

# Search path in Kaggle input or local artifacts
kaggle_split = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = kaggle_split if os.path.exists(kaggle_split) else local_split
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact not found at {kaggle_split} or {local_split}. Run Notebook 01 first.")
print(f"Using split artifact: {split_file}")
    
with open(split_file, "r") as f:
    split_info = json.load(f)

# DataLoaders
train_r_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
train_f_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), train_tf)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

train_r_loader = DataLoader(train_r_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
train_f_loader = DataLoader(train_f_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

all_04a_results = []

for seed in MODEL_SEEDS:
    kaggle_ckpt = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{seed}.pt"
    local_ckpt = f"./artifacts/checkpoints/original_{MODEL_NAME}_seed_{seed}.pt"
    orig_ckpt = kaggle_ckpt if os.path.exists(kaggle_ckpt) else local_ckpt

    kaggle_metrics = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{seed}.json"
    local_metrics = f"./artifacts/metrics/original_{MODEL_NAME}_seed_{seed}.json"
    orig_metrics_file = kaggle_metrics if os.path.exists(kaggle_metrics) else local_metrics
    
    if not os.path.exists(orig_ckpt) or not os.path.exists(orig_metrics_file):
        raise FileNotFoundError(f"Original artifacts for model {MODEL_NAME} seed {seed} missing (checked Kaggle and local paths). Run Notebook 01 first.")
    print(f"Using original ckpt: {orig_ckpt} | metrics: {orig_metrics_file}")
        
    for method_name in METHODS:
        print(f"\n==================================================")
        print(f"--- Running 04a: CMF Static | Method: {method_name} | Model: {MODEL_NAME} | Seed: {seed} ---")
        print(f"==================================================")
        
        # Load fresh original model
        model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
        model.load_state_dict(torch.load(orig_ckpt, map_location=device))
        
        # 1. Reconstruct CMF head on clean D_R and freeze linear head
        reconstruct_cmf_head(model, retain_eval_loader, num_classes=NUM_CLASSES, device=device)
        
        start_time = time.time()
        
        # 2. Run unlearning algorithm on encoder (linear head is frozen)
        if method_name == "random_label":
            unlearned_model = unlearn_random_label(model, train_r_loader, train_f_loader, num_classes=NUM_CLASSES, lr=5e-3, epochs=4, device=device)
        elif method_name == "neggrad_plus":
            unlearned_model = unlearn_neggrad_plus(model, train_r_loader, train_f_loader, alpha=0.5, lr=2e-3, epochs=3, device=device)
        elif method_name == "salun":
            unlearned_model = unlearn_salun(model, train_r_loader, train_f_loader, num_classes=NUM_CLASSES, threshold=0.5, lr=5e-3, epochs=4, device=device)
        elif method_name == "scrub":
            unlearned_model = unlearn_scrub(model, train_r_loader, train_f_loader, epochs=3, msteps=2, lr=5e-3, device=device)
        elif method_name == "tarun_unsir":
            unlearned_model = unlearn_tarun_unsir(model, train_r_loader, forget_loader=train_f_loader, num_classes=NUM_CLASSES, impair_lr=1e-2, repair_lr=5e-3, device=device)
            
        # In static CMF, the CMF head is constructed once before unlearning and remains fixed.
        # The model is evaluated directly with this fixed CMF head.
        runtime = time.time() - start_time
        
        # Save unlearned CMF checkpoint
        save_ckpt = f"./artifacts/checkpoints/cmf_04a_{method_name}_{MODEL_NAME}_seed_{seed}.pt"
        torch.save(unlearned_model.state_dict(), save_ckpt)
        
        # Evaluate 3-tier metrics
        # val_metrics: probe fitted on D_R; output_retain/lp_retain/ncc_retain evaluate D_R;
        #              output_forget evaluates D_F; MIA members=D_F, non-members=D_V.
        val_metrics = evaluate_three_tier_metrics(
            model=unlearned_model, probe_train_loader=retain_eval_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device, nonmember_loader=val_loader
        )
        test_metrics = evaluate_three_tier_metrics(
            model=unlearned_model, probe_train_loader=retain_eval_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device, test_eval_loader=test_loader, nonmember_loader=val_loader
        )
        
        res = {
            "variant": "04a_cmf_static",
            "method": method_name,
            "model_name": MODEL_NAME,
            "seed": seed,
            "split_seed": SPLIT_SEED,
            "runtime_sec": runtime,
            "validation_metrics": val_metrics,
            "test_metrics": test_metrics
        }
        all_04a_results.append(res)
        
        res_path = f"./artifacts/metrics/cmf_04a_{method_name}_{MODEL_NAME}_seed_{seed}.json"
        with open(res_path, "w") as f:
            json.dump(res, f, indent=2)
            
        print(f"[{method_name}] 04a Retain (D_R) Output Acc: {test_metrics['output_retain']:.4f} | Forget (D_F) Output Acc: {test_metrics['output_forget']:.4f} | Test (D_T) Acc: {test_metrics.get('output_test', 0.0):.4f}")
        print(f"[{method_name}] 04a Retain (D_R) LP Acc:     {test_metrics['lp_retain']:.4f} | Forget (D_F) LP Acc:     {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
        print(f"[{method_name}] 04a Retain (D_R) NCC Acc:    {test_metrics['ncc_retain']:.4f} | Forget (D_F) NCC Acc:    {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
        print(f"[{method_name}] Saved 04a checkpoint & metrics to {res_path}")
        
        # Feature space & decision boundary visualization for first seed
        if seed == MODEL_SEEDS[0]:
            vis_save = f"./artifacts/plots/features_04a_cmf_{method_name}_{MODEL_NAME}_seed_{seed}.png"
            visualize_feature_space_and_boundaries(
                model=unlearned_model,
                retain_loader=retain_eval_loader,
                forget_loader=forget_eval_loader,
                num_classes=NUM_CLASSES,
                forget_class=0,
                title=f"Feature Space: 04a CMF Static ({method_name}, {MODEL_NAME})",
                device=device,
                save_path=vis_save
            )

# ============================================================
# Final Consolidated Benchmark Metrics Summary Table
# ============================================================
summary_df = print_metrics_summary_table(
    results=all_04a_results,
    title="04a CMF Static Representation Unlearning Metrics Summary",
    save_csv_path=f"./artifacts/metrics/summary_04a_cmf_{MODEL_NAME}.csv"
)